# Create round_info.csv

Builds `round_info.csv` from the HAL configs (notebook 01) and the FOV/boundary layout (notebook 02) -- the per-round series/HAL-config/data-dir table that notebook 04 turns into the Dave recipe.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave      import create_round_info
from MERci.acquisition.pipeline_config import load_pipeline_config

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale.
PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME is the TRUE top-level experiment id (e.g. "LT058_sample_07"),
# auto-detected from the folder structure -- NOT SAMPLE_DIR.name, which is only
# this acquisition's own local folder name (e.g. "epi") once split into
# sibling acquisition-type subfolders. Must match what notebook 02 used, since
# create_round_info below references positions_{SAMPLE_NAME}.txt.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
# POSITIONS_TAG is what every positions_{...}.txt filename below/passed downstream
# uses -- SAMPLE_NAME alone in the flat layout, or SAMPLE_NAME_IMAGING_DIR split
# layout, so a sibling acquisition never collides.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────
MICROSCOPE  = PIPELINE_CONFIG.microscope   # microscope identifier

# Per-tissue transit-bridging (TISSUE_PATH_MODE, multi-boundary recipe via
# create_round_info_multitissue) is deprecated for now -- see notebook 02's
# own intro cell. Every acquisition uses the single combined positions file
# notebook 02 always writes now, regardless of tissue count.

# HAL config filenames (from notebook 01 / SETTINGS_DIR)
# Adjust these to match the actual files created by notebook 01
bits_hal_configs    = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))

print("Available HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

# Set these manually if auto-detection picks the wrong files
BITS_HAL_CONFIG    = bits_hal_configs[0].name    if bits_hal_configs    else "hal-config-mf3-bits.xml"
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"

print(f"\nBits    HAL config : {BITS_HAL_CONFIG}")
print(f"Cells   HAL config : {CELLS_HAL_CONFIG}")

## Round – bit – color mapping

The round → bit → colour mapping for the codebook is **pipeline-level**:
read from this pipeline's own round_bit_color CSV under
`data/configs/round_bit_color_map/` (pipeline.yaml's
`dataorganization.round_bit_color_csv`), not defined
here. This is the single source of **`N_HYBS`** (the number of
hybridisation/bits rounds, taken as the max round index) used by the recipe
below, and it is saved to `round_bit_color_map.csv` for notebooks 04-07 to
reuse (Dave config, data organization, experiment info, MERlin/fishtank
scripts).

In [ ]:
# round : hyb/bit index (1-indexed), matching the bits movie series number
#         (hal-{mic}_01, _02, …); NOT the Dave imaging-round number.
# bit   : bit number     |     color : excitation wavelength (nm)
# Loaded from this pipeline's own round_bit_color CSV under
# data/configs/round_bit_color_map/ (pipeline.yaml's
# dataorganization.round_bit_color_csv) instead of
# being hardcoded here -- see pipeline_config.py's own docstring.
round_bit_color = PIPELINE_CONFIG.round_bit_color

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

In [ ]:
round_info = create_round_info(
    microscope       = MICROSCOPE,
    n_bits           = N_HYBS,
    bits_hal_config  = BITS_HAL_CONFIG,
    cells_hal_config = CELLS_HAL_CONFIG,
    sample_dir       = SAMPLE_DIR,
    positions_txt    = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt",
)

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")